# GymRAVANA pose model training and evaluation

This notebook compares three real classifiers on MediaPipe joint geometry. Model selection uses source-grouped folds, and the third collage source remains untouched until the selected model is evaluated. The tiny sample size makes every metric highly uncertain.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'artisan').exists())
sys.path.insert(0, str(PROJECT_ROOT))
from ai.pose.workflow import HOLDOUT_SOURCE, evaluate_candidates, load_verified_features

CSV_PATH = PROJECT_ROOT / 'ai/data/pose_features.csv'
METADATA_PATH = PROJECT_ROOT / 'ai/data/pose_features.metadata.json'
REPORT_PATH = PROJECT_ROOT / 'ai/artifacts/pose_model_selection.json'

In [2]:
features, metadata = load_verified_features(CSV_PATH, METADATA_PATH)
print('Verified fingerprint:', metadata['dataset_sha256'])
print('Rows:', len(features), '| classes:', features['canonical_pose'].nunique(), '| sources:', features['source_id'].nunique())
print('Holdout source:', HOLDOUT_SOURCE)
display(pd.crosstab(features['canonical_pose'], features['source_id']))
assert features.groupby(['canonical_pose', 'source_id']).size().eq(1).all()

Verified fingerprint: 1c0de3a8c0d5eb854c2768b3de4d0e844f47e52f5ddf599e968d3acd55191c7c
Rows: 15 | classes: 5 | sources: 3
Holdout source: source_03_correct_examples


source_id,source_01_incorrect_examples,source_02_correct_examples,source_03_correct_examples
canonical_pose,,,
balasana,1,1,1
mayurasana,1,1,1
salamba_sirsasana,1,1,1
urdhva_dhanurasana,1,1,1
virasana,1,1,1


In [3]:
evaluation = evaluate_candidates(CSV_PATH, METADATA_PATH, REPORT_PATH)
print('Selected model:', evaluation['selected_model'])
print('Candidate grouped results:')
display(pd.DataFrame(evaluation['candidates']).T[['mean_accuracy', 'mean_macro_f1']])
print('Untouched source holdout accuracy:', evaluation['holdout']['accuracy'])
print('Untouched source holdout macro F1:', evaluation['holdout']['macro_f1'])
display(pd.DataFrame(evaluation['holdout']['predictions']))

C:\Users\anjan\OneDrive\Documents\Gym-RAVNA\ravana-app\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\anjan\OneDrive\Documents\Gym-RAVNA\ravana-app\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Selected model: random_forest
Candidate grouped results:


,mean_accuracy,mean_macro_f1
logistic_regression,0.4,0.35
random_forest,0.7,0.616667
support_vector_machine,0.6,0.516666


Untouched source holdout accuracy: 1.0
Untouched source holdout macro F1: 1.0


,sample_id,actual,predicted
0,sample_013,virasana,virasana
1,sample_014,balasana,balasana
2,sample_015,urdhva_dhanurasana,urdhva_dhanurasana
3,sample_017,mayurasana,mayurasana
4,sample_018,salamba_sirsasana,salamba_sirsasana


In [4]:
assert evaluation['deployment_allowed'] is False
print('Deployment remains blocked for these reasons:')
for reason in evaluation['deployment_blockers']:
    print('-', reason)

Deployment remains blocked for these reasons:
- Only 15 landmark rows are available; at least 250 are required.
- Only 3 source groups are available; at least 10 are required.
- Every source is an AI/reference collage rather than trainer-verified real participant data.
- There is no trustworthy correct/incorrect form target.


## Metric warning

The held-out set contains only one image per class and is grouped by collage source, not by real participant. These numbers demonstrate executable evaluation code; they are not estimates of production accuracy, fairness, form quality, or injury risk.